# 02 · Map the Collibra terms to the FIBO Product & Account ontology

We align each extracted business term to a **FIBO** class and each relation type
to a **FIBO** object property. The mapping records the alignment strength so the
HBIM builder (notebook 03) knows which axiom to emit:

| kind | axiom |
|------|-------|
| `exact`    | `owl:equivalentClass` |
| `narrower` | `rdfs:subClassOf` |
| `close`    | `skos:closeMatch` |

Reads `output/account_terms_extracted.json`, writes
`output/collibra_to_fibo_mapping.json`.

In [1]:
import sys, json
from pathlib import Path

# Make ../src importable so we can reuse the shared paths + namespaces.
SRC = Path.cwd().parent / "src"
sys.path.insert(0, str(SRC))
from common import (COLLIBRA_EXPORT, EXTRACTED_FILE, MAPPING_FILE,
                    HBIM_TTL, HBIM_INFERRED_TTL, FIBO_EXCERPT, OUTPUT_DIR,
                    PREFIXES, bind_all,
                    CAA, FPAS, FSE, REL, CMNS_ID, CMNS_ORG, HBIM)
import pandas as pd
pd.set_option("display.max_colwidth", 60)
print("Project root:", SRC.parent)

Project root: C:\Users\marci\OneDrive\DEV\EDU\AIML\Graph ML\Ontology Engineering\Ontology Repository\FIBO\Ontology Mapping


In [2]:
with open(EXTRACTED_FILE, encoding="utf-8") as fh:
    extracted = json.load(fh)
pref = extracted["preferred_attributes"]
print("Loaded", len(extracted["business_terms"]), "business terms")

Loaded 8 business terms


## The curated alignment

This is the human-curated alignment a data architect produces when reconciling
the bank glossary with FIBO, expressed as data so the pipeline stays automated.

In [3]:
def C(curie):
    prefix, local = curie.split(":")
    return str(PREFIXES[prefix]) + local

# Business term -> FIBO class
TERM_TO_FIBO = {
    "bt-account":            {"fibo": "caa:Account",                      "kind": "exact"},
    "bt-deposit-account":    {"fibo": "caa:DepositAccount",               "kind": "exact"},
    "bt-current-account":    {"fibo": "caa:DemandDepositAccount",         "kind": "narrower"},
    "bt-savings-account":    {"fibo": "caa:NonTransactionDepositAccount", "kind": "narrower"},
    "bt-account-holder":     {"fibo": "caa:AccountHolder",                "kind": "exact"},
    "bt-account-identifier": {"fibo": "caa:AccountIdentifier",            "kind": "exact"},
    "bt-account-balance":    {"fibo": "caa:Account",                      "kind": "close"},
    "bt-checking-account":   {"fibo": "caa:DemandDepositAccount",         "kind": "close"},
}

# Collibra relation type -> FIBO object property
RELATION_TO_FIBO = {
    "is held by":       "rel:isHeldBy",
    "is identified by": "cmns-id:isIdentifiedBy",
}

## Business term → FIBO class

In [4]:
term_mappings = []
for term in extracted["business_terms"]:
    tid = term["id"]
    m = TERM_TO_FIBO.get(tid)
    if not m:
        continue
    term_mappings.append({
        "collibra_id": tid,
        "business_term": term["name"],
        "preferred_label": pref[tid]["preferred_label"] or term["name"],
        "is_preferred": pref[tid]["is_preferred"],
        "fibo_curie": m["fibo"],
        "fibo_iri": C(m["fibo"]),
        "mapping_kind": m["kind"],
    })

pd.DataFrame(term_mappings)[["business_term", "is_preferred", "mapping_kind", "fibo_curie"]]

,business_term,is_preferred,mapping_kind,fibo_curie
0,Account,True,exact,caa:Account
1,Current Account,True,narrower,caa:DemandDepositAccount
2,Checking Account,False,close,caa:DemandDepositAccount
3,Savings Account,True,narrower,caa:NonTransactionDepositAccount
4,Deposit Account,True,exact,caa:DepositAccount
5,Account Holder,True,exact,caa:AccountHolder
6,Account Balance,True,close,caa:Account
7,Account Identifier,True,exact,caa:AccountIdentifier


## Relation type → FIBO property

In [5]:
relation_mappings = []
for r in extracted["relations"]:
    prop = RELATION_TO_FIBO.get(r["type"])
    if prop:
        relation_mappings.append({
            "collibra_relation": r["type"],
            "source": r["source_id"], "target": r["target_id"],
            "fibo_property_curie": prop, "fibo_property_iri": C(prop),
        })

pd.DataFrame(relation_mappings)[["collibra_relation", "source", "target", "fibo_property_curie"]]

,collibra_relation,source,target,fibo_property_curie
0,is identified by,bt-account,bt-account-identifier,cmns-id:isIdentifiedBy
1,is held by,bt-account,bt-account-holder,rel:isHeldBy


## Persist the mapping

In [6]:
mapping = {
    "category": extracted["category"]["name"],
    "term_mappings": term_mappings,
    "relation_mappings": relation_mappings,
}
with open(MAPPING_FILE, "w", encoding="utf-8") as fh:
    json.dump(mapping, fh, indent=2)
print("[written]", MAPPING_FILE)

[written] C:\Users\marci\OneDrive\DEV\EDU\AIML\Graph ML\Ontology Engineering\Ontology Repository\FIBO\Ontology Mapping\output\collibra_to_fibo_mapping.json
